### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [1]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [2]:
root_data = "/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/colaboracoes/tcga"

gdc = GDC(root_data=root_data)


### All programs

In [3]:
force=False
verbose=True

prog_list = gdc.list_gdc_progams(force=force, verbose=verbose)

# prog_list

File read at '/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/colaboracoes/tcga/programs'


### Get valid subtypes

facets = "diagnoses.primary_diagnosis"

### For each subtype → get stages

facets = "diagnoses.tumor_stage"

### For each (subtype, stage) → get samples

fields = "case_id,submitter_id,diagnoses.tumor_stage"

### Then → fetch RNA-seq files

endpoint = "/files"
field = "cases.case_id"



In [4]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

dfc.head(3)


Table opened ((33, 3)) at '/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/colaboracoes/tcga/primary_site_program_TCGA.tsv'


,project_id,primary_site,disease_type
0,TCGA-ACC,['Adrenal gland'],['Adenomas and Adenocarcinomas']
1,TCGA-BLCA,['Bladder'],"['Epithelial Neoplasms, NOS', 'Squamous Cell N..."
2,TCGA-BRCA,['Breast'],"['Adnexal and Skin Appendage Neoplasms', 'Aden..."


### Subtypes

In [5]:
force=False
verbose=True

i=1
pid = dfc.iloc[i].project_id
primary_site = dfc.iloc[i].primary_site

print(pid, primary_site)

df_subt = gdc.build_subtypes(pid=pid, do_filter=True, force=force, verbose=verbose)

print(len(df_subt))
df_subt

TCGA-BLCA ['Bladder']
Table opened ((22, 6)) at '/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/colaboracoes/tcga/subtype_for_PS_TCGA-BLCA.tsv'
9


,project_id,subtype,n,subtype_raw,is_valid,frac
0,TCGA-BLCA,transitional cell carcinoma,348,transitional cell carcinoma,True,0.701613
1,TCGA-BLCA,papillary transitional cell carcinoma,76,papillary transitional cell carcinoma,True,0.153226
2,TCGA-BLCA,"papillary transitional cell carcinoma, non-inv...",66,"papillary transitional cell carcinoma, non-inv...",True,0.133065
3,TCGA-BLCA,acinar adenocarcinoma,1,acinar adenocarcinoma,True,0.002016
4,TCGA-BLCA,basaloid squamous cell carcinoma,1,basaloid squamous cell carcinoma,True,0.002016
5,TCGA-BLCA,merkel cell carcinoma,1,merkel cell carcinoma,True,0.002016
6,TCGA-BLCA,papillary renal cell carcinoma,1,papillary renal cell carcinoma,True,0.002016
7,TCGA-BLCA,"squamous cell carcinoma, clear cell type",1,"squamous cell carcinoma, clear cell type",True,0.002016
8,TCGA-BLCA,transitional cell carcinoma in situ,1,transitional cell carcinoma in situ,True,0.002016


### Stages

In [8]:
force=False
verbose=True

i=0
subtype = df_subt.iloc[i].subtype_raw


print(pid, subtype)

df_stage = gdc.build_stages(pid=pid, subtype=subtype, force=force, verbose=verbose)

print(len(df_stage))
df_stage

TCGA-BLCA transitional cell carcinoma
Table saved ((14, 7)) at '/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/colaboracoes/tcga/stage_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma.tsv'
13


,project_id,subtype,stage,n,stage_raw,is_valid,frac
0,TCGA-BLCA,transitional cell carcinoma,stage iii,125,stage iii,True,0.301205
1,TCGA-BLCA,transitional cell carcinoma,stage iv,124,stage iv,True,0.298795
2,TCGA-BLCA,transitional cell carcinoma,stage ii,113,stage ii,True,0.272289
3,TCGA-BLCA,transitional cell carcinoma,stage i,24,stage i,True,0.057831
4,TCGA-BLCA,transitional cell carcinoma,stage iia,8,stage iia,True,0.019277
5,TCGA-BLCA,transitional cell carcinoma,stage iib,8,stage iib,True,0.019277
6,TCGA-BLCA,transitional cell carcinoma,stage 0is,4,stage 0is,True,0.009639
7,TCGA-BLCA,transitional cell carcinoma,stage 0a,3,stage 0a,True,0.007229
8,TCGA-BLCA,transitional cell carcinoma,stage ia,2,stage ia,True,0.004819
9,TCGA-BLCA,transitional cell carcinoma,stage 0,1,stage 0,True,0.002410


In [9]:
force=True
verbose=True

i=0
stage = df_stage.iloc[i].stage_raw


print(pid, subtype, stage)

df_cases = gdc.build_cases(pid=pid, subtype=subtype, stage=stage, force=force, verbose=verbose)
print(len(df_cases))
df_cases.head(3)

TCGA-BLCA transitional cell carcinoma stage iii
Table saved ((121, 6)) at '/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/colaboracoes/tcga/stage_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iii.tsv'
121


,project_id,subtype,stage,case_id,n,frac
0,TCGA-BLCA,transitional cell carcinoma,stage iii,aa14e232-c6c6-4075-9d15-e1857373d233,1,0.008264
1,TCGA-BLCA,transitional cell carcinoma,stage iii,e44f60ec-66c4-45a2-a003-0a1faf1d6e03,1,0.008264
2,TCGA-BLCA,transitional cell carcinoma,stage iii,fc0032db-b6d8-4da3-bd1b-beec4196abf0,1,0.008264
